## HateXplain - 2 OBJECTIVES

### Multi Task Learning

* **Dataset:** HateXplain
* **Task :** Classification
* **Target 1:** Hate speech - general
* **Target 2:** Hate speech against woman

In [1]:
%load_ext autoreload 
%autoreload 2

In [2]:
import random
import torch
import numpy as np
from datasets import load_dataset

/home/lineccsa/mestrado/machinemoo/venv_moo/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from machinemoo import moo
from machinemoo import get_objectives, get_models
from machinemoo.analysis.visualization import plot_pareto, plot_multiple_hypervolumes #, plot_pareto_plotly
from machinemoo.analysis.metrics import compute_hypervolume_progress

import preprocessing as pp
from scalarization import NLPScalarization

Set parameter Username
Academic license - for non-commercial use only - expires 2026-04-10
Set parameter Username
Academic license - for non-commercial use only - expires 2026-04-10
Set parameter Username
Academic license - for non-commercial use only - expires 2026-04-10
Set parameter Username
Academic license - for non-commercial use only - expires 2026-04-10
Set parameter Username
Academic license - for non-commercial use only - expires 2026-04-10
Set parameter Username
Academic license - for non-commercial use only - expires 2026-04-10


In [4]:
# Setting a seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed()

## Dataset and Model

In [5]:
# Carregar o dataset HateXplain
dataset = load_dataset("Hate-speech-CNERG/hatexplain", trust_remote_code=True)

# model_name = "tum-nlp/bert-hateXplain"
#"hate-bert, uncased bert"
model_name="bert-base-uncased"

In [6]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda:0


In [7]:
task_config = {
    "hate_speech": {"type": "hate_speech"},
    "women": {"type": "group_hate", "group": "Women"},
    #"homosexual": {"type": "group_hate", "group": "Homosexual"},
    #"indigenous": {"type": "group_hate", "group": "Indigenous"},
    #"african": {"type": "group_hate", "group": "African"},
    #"asian": {"type": "group_hate", "group": "Asian"},
    #"jewish": {"type": "group_hate", "group": "Jewish"}
}

train_dataloader = pp.get_dataloader(dataset['train'], task_config)

100%|██████████| 481/481 [02:32<00:00,  3.15it/s]


In [8]:
import torch.nn as nn
from tqdm import tqdm

In [9]:
class MultiTaskModel(nn.Module):
    def __init__(
        self,
        task_names: list[str],
        input_dim: int = 50,
        hidden_dim: int = 32,
        num_labels: int = 2,
        dropout_rate: float = 0.3
    ) -> None:
        super(MultiTaskModel, self).__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.shared_fc = nn.Linear(input_dim, hidden_dim)

        # Task-specific heads stored in a ModuleDict
        self.classifiers = nn.ModuleDict(
            {task: nn.Linear(hidden_dim, num_labels) for task in task_names}
        )

    def forward(self, embeddings):
        x = self.dropout(embeddings)
        features = self.shared_fc(x)

        # Compute logits for each task and return a dict
        return {task: head(features) for task, head in self.classifiers.items()}

In [10]:
task_names = list(task_config.keys())

In [11]:

model = MultiTaskModel(task_names=task_names).to(device)


In [12]:
dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'annotators', 'rationales', 'post_tokens'],
        num_rows: 15383
    })
    validation: Dataset({
        features: ['id', 'annotators', 'rationales', 'post_tokens'],
        num_rows: 1922
    })
    test: Dataset({
        features: ['id', 'annotators', 'rationales', 'post_tokens'],
        num_rows: 1924
    })
})

In [13]:
valid_dataloader = pp.get_dataloader(dataset['validation'], task_config)

100%|██████████| 61/61 [00:13<00:00,  4.49it/s]


In [14]:
optimizer = torch.optim.AdamW(
model.parameters(), lr=2e-5, weight_decay=0.01)
criterion = nn.CrossEntropyLoss()

model.train()
for epoch in range(3):
    total_loss = 0
    for batch in tqdm(
        train_dataloader, desc=f"Epoch {epoch+1}/{3}"
    ):
        embeddings = batch["embedding"].to(device)
        logits_dict = model(embeddings)
        optimizer.zero_grad()

        losses = []
        for i, task in enumerate(task_names):
            labels = batch[task].to(device)
            task_loss = criterion(logits_dict[task], labels)
            losses.append(task_loss)

        loss = sum(losses) / 2
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(
        f"Epoch {epoch+1} - Average loss: {total_loss / len(train_dataloader):.4f}"
    )
    print(f"Epoch {epoch+1} - Loss1: {losses[0].item():.4f} - Loss2: {losses[1].item():.4f}")

# Final evaluation
total_batches = 0
eval_losses = np.zeros(2)
model.eval()
criterion = nn.CrossEntropyLoss()
with torch.no_grad():
    for batch in train_dataloader:
        embeddings = batch["embedding"].to(device)
        logits_dict = model(embeddings)
        for i, task in enumerate(task_names):
            labels = batch[task].to(device)
            loss = criterion(logits_dict[task], labels)
            eval_losses[i] += loss.item()
        total_batches += 1
print("eval:", [loss / total_batches for loss in eval_losses])

Epoch 1/3: 100%|██████████| 241/241 [00:04<00:00, 57.04it/s]


Epoch 1 - Average loss: 0.7691
Epoch 1 - Loss1: 0.7580 - Loss2: 0.8817


Epoch 2/3: 100%|██████████| 241/241 [00:04<00:00, 54.27it/s]


Epoch 2 - Average loss: 0.7504
Epoch 2 - Loss1: 0.6843 - Loss2: 0.7913


Epoch 3/3: 100%|██████████| 241/241 [00:03<00:00, 61.82it/s]


Epoch 3 - Average loss: 0.7339
Epoch 3 - Loss1: 0.6992 - Loss2: 0.7003
eval: [np.float64(0.7005020708958638), np.float64(0.7421122900677914)]


In [ ]:
# 3, 4, 7 tasks